### Purpose

1. 이카운트 품목 전체를 받는다 (Zone → 로그인 → 품목조회)
2. 약품 코드(A·B·C·G로 시작)만 요약한다
3. 현재 약품 DB(dschemicals)와 비교해 신규 품목을 미리 본다
4. 웹(약품목록 > 이카운트 비교)이 읽는 `s3://dsbaseinfo/ecount/products_latest.json` 에 올린다 (Lambda `dsecountProducts` → `GET /ecountproducts` 로 제공)

- 이카운트 실서버 키는 **등록된 IP에서만** 로그인된다 → 사무실 PC에서 실행
- 인증 정보는 같은 폴더의 `.env` (git 제외)
- 이카운트 호출 한도: 하루 5,000회, 시간당 연속 오류 30회

In [1]:
# Norton 등이 HTTPS를 가로채는 PC에서 SSL 오류 방지 - boto3/requests 보다 먼저 적용해야 함
import truststore
truststore.inject_into_ssl()

import os
import json
import urllib.request
from datetime import datetime

import pandas as pd
import boto3

# fetch_products.py 의 설정/함수 재사용
from fetch_products import (load_dotenv, summarize, HERE, DATA_DIR,
                            S3_BUCKET, S3_KEY, AWS_PROFILE, PREFIXES)
from ecount_client import EcountClient, EcountSettings

boto3.setup_default_session(profile_name=AWS_PROFILE)
pd.set_option('display.max_rows', 100)

Settings

In [2]:
DSCHEMICAL_API = 'https://jyipsj28s9.execute-api.us-east-1.amazonaws.com/dev/dschemical'

load_dotenv(HERE / '.env')
settings = EcountSettings.from_env()
settings.env, settings.com_code   # 인증키는 출력하지 않음

('prod', '610207')

이카운트 품목 받기

In [3]:
client = EcountClient(settings)
items, resp = client.get_products()
data = resp.get('Data') or {}
fetched_at = datetime.now().astimezone().isoformat(timespec='seconds')

print(f"이카운트 품목 {len(items)}건 (TotalCnt={data.get('TotalCnt')})")
print(f"호출 한도: {data.get('QUANTITY_INFO', '')}")

이카운트 품목 1376건 (TotalCnt=1376)
호출 한도: 시간당 연속 오류 제한 건수 : 0/30건, 1일 허용량 : 12/5000건


약품 접두어만 요약

In [4]:
rows = summarize(items)
payload = {
    'fetched_at': fetched_at,
    'total_cnt': len(items),
    'prefixes': list(PREFIXES),
    'quantity_info': data.get('QUANTITY_INFO', ''),
    'items': rows,
    # 모든 품목코드 → 품목명 (웹에서 '여기서 만든 약품'의 이카운트 등록 여부·코드 충돌 확인용)
    'all_codes': {(i.get('PROD_CD') or '').strip(): (i.get('PROD_DES') or '').strip() for i in items},
}

df_ec = pd.DataFrame(rows)
df_ec['prefix'] = df_ec['PROD_CD'].str[:2]
print(f"비교 대상 {len(df_ec)}건")
df_ec['prefix'].value_counts().sort_index()

비교 대상 656건


prefix
A1    169
A2     60
A3     71
B0    291
C0     49
G0     16
Name: count, dtype: int64

현재 약품 DB와 비교 (신규 = 이카운트에만 있음)

In [5]:
with urllib.request.urlopen(DSCHEMICAL_API) as r:
    res_ = json.load(r)
body = res_.get('body', res_) if isinstance(res_, dict) else res_
db = json.loads(body) if isinstance(body, str) else body

db_codes = {c['dsids'] for c in db}
df_new = df_ec[~df_ec['PROD_CD'].isin(db_codes)]
db_only = sorted(db_codes - set(df_ec['PROD_CD']))

print(f"약품 DB {len(db_codes)}건 / 신규 {len(df_new)}건 / DB에만 있음 {len(db_only)}건 {db_only}")
df_new[['PROD_CD', 'PROD_DES', 'SIZE_DES', 'PROD_TYPE', 'IN_PRICE']]

약품 DB 630건 / 신규 28건 / DB에만 있음 2건 ['B50381', 'B50391']


,PROD_CD,PROD_DES,SIZE_DES,PROD_TYPE,IN_PRICE
126,A15812,퀸텍,250㎖,1,0
128,A15822,트위터,200㎖,1,0
168,A18001,청풍명월,250㎖,1,0
299,A38021,레인그란,500㎖,1,0
422,B04552,스마트로NPK,20kg,1,0
577,B07691,입상부식산,20kg,1,0
586,B07940,루트칼,1kg,1,0
587,B07960,바이오부스트,1ℓ,1,0
588,B07981,노틸러스,4.5ℓ,1,0
589,B07991,크리스탈그린,20kg,1,0


로컬 저장 (원본은 구매처 등 포함 → `data/` 는 git 제외)

In [6]:
DATA_DIR.mkdir(exist_ok=True)
stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
raw_path = DATA_DIR / f'products_raw_{stamp}.json'
raw_path.write_text(json.dumps({'fetched_at': fetched_at, 'items': items}, ensure_ascii=False), encoding='utf-8')
(DATA_DIR / 'products_latest.json').write_text(json.dumps(payload, ensure_ascii=False, indent=1), encoding='utf-8')
print('로컬 저장:', raw_path.name, ', products_latest.json')

로컬 저장: products_raw_20261001_010132.json , products_latest.json


S3 업로드 → 웹의 이카운트 비교 버튼에 반영

In [7]:
s3 = boto3.client('s3')
s3.put_object(
    Bucket=S3_BUCKET,
    Key=S3_KEY,
    Body=json.dumps(payload, ensure_ascii=False).encode('utf-8'),
    ContentType='application/json; charset=utf-8',
    CacheControl='no-cache',
)
head = s3.head_object(Bucket=S3_BUCKET, Key=S3_KEY)
print(f"S3 업로드: s3://{S3_BUCKET}/{S3_KEY} ({head['ContentLength']:,} bytes, {head['LastModified']})")

S3 업로드: s3://dsbaseinfo/ecount/products_latest.json (89,751 bytes, 2026-10-01 05:01:37+00:00)
